# NetGuard ML — CIC-IDS2017 Binary Intrusion Classification

**Phases 2–3: data exploration, preprocessing, binary model training, and evaluation.** The objective is to classify flows as `BENIGN` (0) or `ATTACK` (1; all observed non-BENIGN source labels).

This notebook uses a user-configurable local CSV and is currently configured for `data/raw/Wednesday-workingHours.pcap_ISCX.csv` as the modeling subset. It does not download data. Model scores and artifacts are produced only when the notebook runs successfully on the real source file.

Dataset source: [official CIC-IDS2017 page](https://www.unb.ca/cic/datasets/ids-2017.html). See `PROJECT_SPEC.md` for scope, data plan, evaluation, and limitations.

## Phase 2 — Data exploration and preprocessing

The local `data/raw/` folder now contains eight CIC-IDS2017 daily/subset CSVs. For this run, the configuration cell explicitly selects `Wednesday-workingHours.pcap_ISCX.csv`; the other files are not included in this model run. The selected input is streamed in chunks and reduced to a reproducible, class-proportional sample with a fixed upper bound. No data is downloaded or synthesized.

The workflow audits the actual header, normalized column names, data types, source labels, missing values, and infinities. It maps `BENIGN` to 0 and each observed non-empty non-BENIGN label to 1, while displaying the exact source labels for review. Rows without a target or with no usable input feature are removed. Duplicate records are counted in the bounded sample but retained: identical flow rows can be legitimate, and there is no evidence yet to justify global duplicate removal. Identifier, timestamp, and label/post-event-like fields are excluded; ports are retained as candidate flow features for later leakage review.

A fixed-seed stratified random split is the baseline. Related flows can occur in both partitions, so this split may overestimate generalization; a file/day-aware holdout should be assessed later. Numeric and categorical preprocessing is fitted on training data only. The test set is not used for feature selection or tuning.

Each code cell is followed by markdown interpreting its output. Figures are saved under `outputs/figures/`. Results and artifacts must come from a successful run on this selected real CSV.

### 1. Configure the local source and reproducibility settings

Set `CIC_IDS2017_CSV` to override the selected file, or edit `DATA_PATH` below. The current default is `data/raw/Wednesday-workingHours.pcap_ISCX.csv`, one of the files present in the project. `CHUNK_SIZE` bounds each CSV read, and the class-proportional sample is capped at `MAX_SAMPLE_ROWS`. No remote downloads occur.

In [ ]:
from collections import Counter
import math
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Use the Wednesday daytime CIC-IDS2017 CSV as the explicitly selected subset.
DATA_PATH = Path(os.environ.get("CIC_IDS2017_CSV", "data/raw/Wednesday-workingHours.pcap_ISCX.csv")).expanduser()
CHUNK_SIZE = 100_000
MAX_SAMPLE_ROWS = 50_000
RANDOM_STATE = 42
TEST_SIZE = 0.20
FIGURE_DIR = Path("outputs/figures")

if CHUNK_SIZE < 1 or MAX_SAMPLE_ROWS < 2:
    raise ValueError("CHUNK_SIZE must be positive and MAX_SAMPLE_ROWS must be at least 2.")

print(f"Configured CSV: {DATA_PATH.resolve()}")
print(f"File exists: {DATA_PATH.is_file()}")
print(f"Chunk size: {CHUNK_SIZE:,}; maximum retained sample: {MAX_SAMPLE_ROWS:,}")
print(f"Random seed: {RANDOM_STATE}; test fraction: {TEST_SIZE:.0%}")
print(f"Figure output directory: {FIGURE_DIR.resolve()}")

The configuration output identifies the exact local CSV path and whether it exists. If it does not, change `DATA_PATH` or set `CIC_IDS2017_CSV`; the loading step raises an explicit `FileNotFoundError` rather than downloading data or substituting made-up records. The chunk size and sample cap make the memory trade-off visible and reproducible.

### 2. Inspect the schema, audit data quality, and build a bounded sample

Column names are normalized to lowercase snake case; collisions after normalization are rejected because they would make feature selection ambiguous. The target is identified from the normalized `label`, `class`, `target`, or `attack_label` names, and ambiguous/missing target columns are errors. The file is read incrementally. A seeded random-priority reservoir is maintained separately for the two binary classes, then sampled in proportion to the eligible source class counts. This both bounds memory and preserves both classes when both occur and the configured sample cap allows it.

The audit counts source missing and numeric infinite values before cleaning. Whitespace-only labels count as missing. Infinite feature values are changed to missing in the retained sample so the training-only imputer can handle them. Rows missing a label and rows with every remaining feature unusable are excluded. Duplicate rows are not deleted.

In [ ]:
def normalize_column_names(columns):
    normalized = []
    for column in columns:
        name = str(column).replace("\ufeff", "").strip().lower()
        name = re.sub(r"[^a-z0-9]+", "_", name).strip("_")
        if not name:
            raise ValueError(f"A source column name cannot be normalized safely: {column!r}")
        normalized.append(name)
    if len(set(normalized)) != len(normalized):
        duplicates = sorted({name for name in normalized if normalized.count(name) > 1})
        raise ValueError(f"Column names collide after normalization: {duplicates}")
    return normalized


def exclusion_reason(column, target_column):
    if column == target_column:
        return "target label"
    if (
        column in {"flow_id", "flowid", "source_ip", "destination_ip", "src_ip", "dst_ip"}
        or column.endswith("_ip")
    ):
        return "identifier / IP address"
    if (
        "timestamp" in column
        or column in {"date", "datetime", "start_time", "end_time", "flow_start", "flow_end"}
        or column.endswith("_date")
        or column.endswith("_time")
    ):
        return "raw timestamp / date"
    if any(token in column for token in ("label", "target", "class", "attack", "outcome", "prediction")):
        return "label-like or potential post-event leakage"
    return None


if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"CIC-IDS2017 CSV not found: {DATA_PATH.resolve()}. "
        "Set CIC_IDS2017_CSV or edit DATA_PATH to a locally available CSV."
    )

raw_header = pd.read_csv(DATA_PATH, nrows=0, encoding="utf-8-sig").columns
normalized_headers = normalize_column_names(raw_header)
target_candidates = [
    name for name in normalized_headers if name in {"label", "class", "target", "attack_label"}
]
if len(target_candidates) != 1:
    raise ValueError(
        "Expected exactly one recognizable target column named Label, Class, Target, "
        f"or Attack Label after normalization; found {target_candidates}. Source headers: {list(raw_header)}"
    )
target_column = target_candidates[0]

exclusion_reasons = {
    name: reason
    for name in normalized_headers
    if (reason := exclusion_reason(name, target_column)) is not None
}
feature_columns = [name for name in normalized_headers if name not in exclusion_reasons]
if not feature_columns:
    raise ValueError("No usable candidate feature columns remain after excluding target/leakage fields.")

rng = np.random.default_rng(RANDOM_STATE)
source_label_counts = Counter()
eligible_class_counts = Counter()
class_reservoirs = {0: pd.DataFrame(), 1: pd.DataFrame()}
source_missing_counts = pd.Series(0, index=normalized_headers, dtype="int64")
source_infinite_counts = pd.Series(0, index=normalized_headers, dtype="int64")
source_dtype_summary = None
total_source_rows = 0
missing_target_rows = 0
all_features_unusable_rows = 0

for raw_chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
    on_bad_lines="error",
):
    chunk = raw_chunk.copy()
    chunk.columns = normalize_column_names(chunk.columns)
    if list(chunk.columns) != normalized_headers:
        raise ValueError("CSV columns changed between the header and a data chunk.")
    if source_dtype_summary is None:
        source_dtype_summary = chunk.dtypes.astype(str)

    total_source_rows += len(chunk)
    missing_in_chunk = chunk.isna().sum()
    for column in chunk.select_dtypes(include=["object", "string"]).columns:
        blank_count = chunk[column].astype("string").str.strip().eq("").fillna(False).sum()
        missing_in_chunk[column] += int(blank_count)
    source_missing_counts = source_missing_counts.add(missing_in_chunk, fill_value=0).astype("int64")
    numeric_source_columns = chunk.select_dtypes(include=[np.number]).columns
    if len(numeric_source_columns):
        chunk_infinities = np.isinf(chunk[numeric_source_columns]).sum().astype("int64")
        source_infinite_counts.loc[chunk_infinities.index] += chunk_infinities
    for column in chunk.select_dtypes(include=["object", "string"]).columns:
        values = chunk[column].astype("string").str.strip().str.lower()
        string_infinities = values.isin({"inf", "+inf", "-inf", "infinity", "+infinity", "-infinity"})
        source_infinite_counts[column] += int(string_infinities.sum())

    labels = chunk[target_column].astype("string").str.strip().str.upper()
    missing_label_mask = labels.isna() | labels.eq("")
    missing_target_rows += int(missing_label_mask.sum())
    valid_labels = labels.loc[~missing_label_mask]
    source_label_counts.update(valid_labels.value_counts().to_dict())

    if valid_labels.empty:
        continue
    valid_features = chunk.loc[valid_labels.index, feature_columns].copy()
    valid_features = valid_features.replace([np.inf, -np.inf], np.nan)
    for column in valid_features.select_dtypes(include=["object", "string"]).columns:
        valid_features[column] = valid_features[column].replace(r"^\s*$", np.nan, regex=True)
    usable_mask = valid_features.notna().any(axis=1)
    all_features_unusable_rows += int((~usable_mask).sum())
    valid_features = valid_features.loc[usable_mask]
    valid_labels = valid_labels.loc[usable_mask]
    binary_targets = np.where(valid_labels.eq("BENIGN").to_numpy(), 0, 1).astype("int8")

    for class_id in (0, 1):
        class_mask = binary_targets == class_id
        class_size = int(class_mask.sum())
        if class_size == 0:
            continue
        eligible_class_counts[class_id] += class_size
        class_rows = valid_features.loc[class_mask].copy()
        class_rows["target"] = class_id
        class_rows["__sample_priority"] = rng.random(class_size)
        combined = pd.concat([class_reservoirs[class_id], class_rows], ignore_index=True)
        class_reservoirs[class_id] = combined.nsmallest(
            MAX_SAMPLE_ROWS, "__sample_priority"
        ).reset_index(drop=True)

if total_source_rows == 0:
    raise ValueError(f"The selected CSV contains no data rows: {DATA_PATH.resolve()}")
total_eligible_rows = sum(eligible_class_counts.values())
if total_eligible_rows == 0:
    raise ValueError("No rows have both a non-empty label and at least one usable input feature.")

present_classes = [class_id for class_id in (0, 1) if eligible_class_counts[class_id] > 0]
sample_size = min(MAX_SAMPLE_ROWS, total_eligible_rows)
if sample_size < len(present_classes):
    raise ValueError("The sample cap is too small to retain every observed class; increase MAX_SAMPLE_ROWS.")
quotas = {
    class_id: sample_size * eligible_class_counts[class_id] / total_eligible_rows
    for class_id in present_classes
}
sample_allocation = {class_id: int(math.floor(quotas[class_id])) for class_id in present_classes}
for class_id in present_classes:
    if sample_allocation[class_id] == 0:
        donor = max(present_classes, key=lambda candidate: sample_allocation[candidate])
        if sample_allocation[donor] > 0:
            sample_allocation[donor] -= 1
        sample_allocation[class_id] = 1
while sum(sample_allocation.values()) < sample_size:
    eligible_donors = [
        class_id
        for class_id in present_classes
        if sample_allocation[class_id] < eligible_class_counts[class_id]
    ]
    if not eligible_donors:
        break
    chosen = max(
        eligible_donors,
        key=lambda class_id: (quotas[class_id] - sample_allocation[class_id], eligible_class_counts[class_id]),
    )
    sample_allocation[chosen] += 1

sampled = pd.concat(
    [
        class_reservoirs[class_id].nsmallest(
            sample_allocation[class_id], "__sample_priority"
        )
        for class_id in present_classes
    ],
    ignore_index=True,
).drop(columns="__sample_priority")
sampled = sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
sampled["target"] = sampled["target"].astype("int8")
sample_duplicate_rows = int(sampled.duplicated().sum())

print(f"Source file: {DATA_PATH.resolve()} ({DATA_PATH.stat().st_size:,} bytes)")
print(f"Full source shape: ({total_source_rows:,}, {len(normalized_headers):,})")
print(f"Retained sample shape: {sampled.shape}; cap: {MAX_SAMPLE_ROWS:,}")
print(f"Detected target column: {target_column!r}")
print(f"Rows removed for missing/blank target: {missing_target_rows:,}")
print(f"Rows removed because all candidate features were unusable: {all_features_unusable_rows:,}")
print(f"Duplicate rows in bounded sample (retained): {sample_duplicate_rows:,}")
print("Normalized source columns:")
print(normalized_headers)
print("Excluded columns and reasons:")
display(pd.DataFrame(exclusion_reasons.items(), columns=["column", "reason"]))
print("Observed non-empty source label distribution (after whitespace normalization):")
display(pd.Series(source_label_counts, name="source_rows").sort_values(ascending=False).to_frame())
print("Binary class counts in eligible source rows and retained sample:")
class_audit = pd.DataFrame(
    {
        "eligible_source_rows": {class_id: eligible_class_counts[class_id] for class_id in (0, 1)},
        "sample_rows": sampled["target"].value_counts().reindex([0, 1], fill_value=0),
    }
).rename(index={0: "BENIGN (0)", 1: "ATTACK (1)"})
display(class_audit)

source_dtype_summary = source_dtype_summary.reindex(normalized_headers)
source_audit = pd.DataFrame(
    {
        "first_chunk_dtype": source_dtype_summary,
        "source_missing_values": source_missing_counts,
        "source_infinite_values": source_infinite_counts,
    }
)
print("Source columns, inferred type in the first chunk, and full-file missing/infinite counts:")
display(source_audit)
print("Retained sample feature data types:")
display(sampled.drop(columns="target").dtypes.rename("sample_dtype").to_frame())
print("Observed non-BENIGN source labels mapped to ATTACK (1):")
print(sorted(label for label in source_label_counts if label != "BENIGN"))
print("All non-empty non-BENIGN source labels above are mapped to ATTACK per the approved binary scope; review this list for unexpected values.")

Interpret the audit above using the values produced by the configured file: source shape is the streamed full-file row/column count, while the retained shape is the bounded working sample. The audit displays every normalized source column, a first-chunk dtype (CSV inference may vary between chunks), full-source missing/infinite counts, raw source-label distribution, mapped class counts, excluded-field reasons, and sample dtypes. An empty target is excluded; infinite numeric input values are converted to missing before modeling. The exact non-BENIGN labels are shown rather than hidden behind the binary mapping. The duplicate count is for the sample only and those records are retained.

### 3. Visualize class balance and source data-quality counts

The first figure makes the retained sample's BENIGN/ATTACK balance visible. The second summarizes columns with missing or infinite source values (up to 20 columns, ordered by issue count). Both figures are displayed inline and saved for later reporting. A clean audit is explicitly shown rather than replaced with invented values.

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sample_class_counts = (
    sampled["target"]
    .map({0: "BENIGN", 1: "ATTACK"})
    .value_counts()
    .reindex(["BENIGN", "ATTACK"], fill_value=0)
)
fig, ax = plt.subplots(figsize=(7, 4))
sample_class_counts.plot(kind="bar", color=["#4C78A8", "#E45756"], ax=ax)
ax.set_title("CIC-IDS2017 retained sample: binary class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Rows")
ax.tick_params(axis="x", labelrotation=0)
fig.tight_layout()
label_figure_path = FIGURE_DIR / "label_distribution.png"
fig.savefig(label_figure_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

quality_audit = pd.DataFrame(
    {"Missing values": source_missing_counts, "Infinite values": source_infinite_counts}
)
quality_audit = quality_audit.loc[quality_audit.sum(axis=1) > 0]
if quality_audit.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.text(0.5, 0.5, "No missing or infinite source values observed", ha="center", va="center")
    ax.set_axis_off()
else:
    quality_audit["total_issues"] = quality_audit.sum(axis=1)
    quality_audit = quality_audit.sort_values("total_issues").tail(20)
    fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(quality_audit))))
    quality_audit[["Missing values", "Infinite values"]].plot(
        kind="barh", ax=ax, color=["#F2CF5B", "#B279A2"]
    )
    ax.set_title("Source missing and infinite values (top 20 affected columns)")
    ax.set_xlabel("Count across selected CSV")
    ax.set_ylabel("Normalized source column")
    ax.legend(title="Data-quality issue")
fig.tight_layout()
quality_figure_path = FIGURE_DIR / "missing_and_infinite_values.png"
fig.savefig(quality_figure_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved class-distribution figure: {label_figure_path.resolve()}")
print(f"Saved data-quality figure: {quality_figure_path.resolve()}")

The class plot reports only the bounded sample, so it should be interpreted alongside the full-file label counts above. The data-quality plot counts source missing values and numeric infinities before imputation; the imputer will not erase the evidence in this audit. Both PNGs are saved under `outputs/figures/`. No chart values are asserted until the notebook is run against the selected local file.

### 4. Stratified train/test split and training-only preprocessing

The target is separated from the input matrix before splitting. A reproducible stratified random 80/20 split is used only if both classes have enough records for both partitions. A training-fitted `ColumnTransformer` imputes numeric columns with the training median and standardizes them; categorical values are imputed with a missing-value token and one-hot encoded with a 100-category cap to bound expansion. Unseen categorical values at inference are ignored by the fitted encoder. Feature names and order are retained in `feature_columns`; `prepare_features_for_inference` applies the same normalized schema and numeric infinity handling before `preprocessor.transform`.

This random row split is a baseline, not a leakage-proof estimate: related CIC-IDS2017 flows can cross train/test. A later evaluation should include a file/day-aware holdout. This phase does not fit a classifier or use test records to select features.

In [ ]:
X = sampled.loc[:, feature_columns].copy()
y = sampled["target"].astype("int8")
sample_class_counts = y.value_counts().reindex([0, 1], fill_value=0)
if set(y.unique()) != {0, 1}:
    raise ValueError(
        "The retained sample does not contain both BENIGN and ATTACK rows. "
        "Choose a source/subset with both classes or increase MAX_SAMPLE_ROWS."
    )
if int(sample_class_counts.min()) < 2:
    raise ValueError("Each class needs at least two sampled rows for a stratified train/test split.")
expected_test_rows = math.ceil(len(y) * TEST_SIZE)
if expected_test_rows < 2 or len(y) - expected_test_rows < 2:
    raise ValueError(
        "The sample is too small to put both classes in train and test; "
        "increase the source subset or adjust the documented sample size."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
if set(y_train.unique()) != {0, 1} or set(y_test.unique()) != {0, 1}:
    raise RuntimeError("Stratified split did not preserve both classes in both partitions.")

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [column for column in X_train.columns if column not in numeric_features]
if not numeric_features and not categorical_features:
    raise ValueError("No input features are available for preprocessing.")

transformers = []
if numeric_features:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]
    )
    transformers.append(("numeric", numeric_pipeline, numeric_features))
if categorical_features:
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=100)),
        ]
    )
    transformers.append(("categorical", categorical_pipeline, categorical_features))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=True,
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
processed_feature_names = preprocessor.get_feature_names_out().tolist()

def matrix_values_are_finite(matrix):
    values = matrix.data if hasattr(matrix, "tocsr") else np.asarray(matrix)
    return bool(np.isfinite(values).all())


if not matrix_values_are_finite(X_train_processed) or not matrix_values_are_finite(X_test_processed):
    raise ValueError("Non-finite values remain after training-fitted preprocessing.")

def prepare_features_for_inference(incoming_frame):
    """Align new records to the training feature schema before preprocessor.transform."""
    normalized = normalize_column_names(incoming_frame.columns)
    aligned = incoming_frame.copy()
    aligned.columns = normalized
    missing_columns = [column for column in feature_columns if column not in aligned.columns]
    if missing_columns:
        raise ValueError(f"Inference data is missing required feature columns: {missing_columns}")
    aligned = aligned.loc[:, feature_columns].copy()
    for column in numeric_features:
        aligned[column] = pd.to_numeric(aligned[column], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        )
    for column in categorical_features:
        aligned[column] = aligned[column].astype("object").replace(r"^\s*$", np.nan, regex=True)
    return aligned

print(f"Input feature columns ({len(feature_columns)}), in fixed order:")
print(feature_columns)
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print("Class counts by partition:")
split_class_counts = pd.DataFrame(
    {
        "train": y_train.value_counts().reindex([0, 1], fill_value=0),
        "test": y_test.value_counts().reindex([0, 1], fill_value=0),
    }
).rename(index={0: "BENIGN (0)", 1: "ATTACK (1)"})
display(split_class_counts)
print(f"Numeric features: {len(numeric_features)}; categorical features: {len(categorical_features)}")
print(f"Transformed train shape: {X_train_processed.shape}; test shape: {X_test_processed.shape}")
print(f"All transformed values finite: {matrix_values_are_finite(X_train_processed) and matrix_values_are_finite(X_test_processed)}")
print("Preprocessing fitted on X_train only; X_test was transformed without fitting.")
print(f"Inference preparation uses {len(feature_columns)} fixed input columns and {len(processed_feature_names)} transformed columns.")

The split table must show both BENIGN and ATTACK in train and test for this baseline to proceed. The printed feature list is the ordered inference contract: the target and excluded identifier/time/label-like columns are absent. Transformed train and test shapes should have the same number of columns, and the finite-value check should be `True`. Missing-value statistics and class counts are computed from the real selected CSV; no model is trained in this phase. If a guard fails, investigate the source/subset and sample size rather than silently changing labels or replacing data.

## Phase 2 summary

This notebook implements a configurable, local-only CIC-IDS2017 exploration and preprocessing workflow: chunked reading, normalized schema and label handling, full-file quality counts, a reproducible bounded class-proportional sample, explicit exclusions and cleaning decisions, saved exploratory figures, and a stratified baseline split with train-only preprocessing. Exact row counts, labels, data-quality totals, split counts, and transformed shapes must be taken from the preceding output of the user's configured dataset; this notebook preparation step has not loaded or fabricated any CIC-IDS2017 records. The random split may overestimate generalization when related flows cross partitions. The resulting train/test matrices are ready for a later model-training phase, not a claim of detection performance or production readiness.

### 1. Train Logistic Regression and Random Forest

Both classifiers use the same Phase 2 training/test split. A clone of the preprocessing transformer is placed inside each saved scikit-learn pipeline; fitting the pipeline learns imputation, scaling, and encoding from `X_train` only. `X_test` is used only to compute final predictions and metrics.

Class imbalance is handled with estimator class weights, not synthetic/duplicated rows: Logistic Regression uses `class_weight="balanced"`; Random Forest uses `class_weight="balanced_subsample"`. Seeds are fixed where supported. The selection rule prioritizes ATTACK-class F1, then ATTACK recall, then lower false-positive rate; Logistic Regression wins only an exact tie as the simpler model.

False positives are benign flows flagged for investigation; false negatives are attacks missed as benign. Metrics are calculated from genuine held-out predictions. No test-driven tuning occurs.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

if not {0, 1}.issubset(set(y_train.unique())) or not {0, 1}.issubset(set(y_test.unique())):
    raise ValueError("Both BENIGN (0) and ATTACK (1) must be present in train and test before model training.")

model_estimators = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

fitted_pipelines = {}
model_predictions = {}
model_results = {}
classification_reports = {}
for model_name, estimator in model_estimators.items():
    fitted_pipeline = Pipeline(
        steps=[
            ("preprocessing", clone(preprocessor)),
            ("classifier", estimator),
        ]
    )
    fitted_pipeline.fit(X_train, y_train)
    predictions = fitted_pipeline.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    classification_reports[model_name] = classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["BENIGN", "ATTACK"],
        zero_division=0,
        digits=4,
    )
    model_predictions[model_name] = predictions
    fitted_pipelines[model_name] = fitted_pipeline
    model_results[model_name] = {
        "accuracy": float(accuracy_score(y_test, predictions)),
        "precision": float(precision_score(y_test, predictions, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_test, predictions, pos_label=1, zero_division=0)),
        "f1_score": float(f1_score(y_test, predictions, pos_label=1, zero_division=0)),
        "macro_f1_score": float(f1_score(y_test, predictions, average="macro", zero_division=0)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

# Prioritize attack F1, then attack recall, then lower FPR; prefer logistic regression only for an exact tie.
selected_model_name = max(
    model_results,
    key=lambda name: (
        model_results[name]["f1_score"],
        model_results[name]["recall"],
        -model_results[name]["false_positive_rate"],
        name == "Logistic Regression",
    ),
)
selected_pipeline = fitted_pipelines[selected_model_name]

results_table = pd.DataFrame(model_results).T[
    ["accuracy", "precision", "recall", "f1_score", "macro_f1_score", "false_positive_rate"]
]
print("Held-out test metrics (positive class: ATTACK=1):")
display(results_table.style.format("{:.4f}"))
for model_name, report in classification_reports.items():
    print(f"Classification report — {model_name} (zero_division=0):")
    print(report)
print(f"Selected model by attack F1, then recall, then lower FPR (exact tie: Logistic Regression): {selected_model_name}")
print("Confusion counts use [true negatives, false positives, false negatives, true positives].")

The results table and classification reports above are computed on the held-out test labels and the predictions produced by each fitted pipeline; `zero_division=0` makes undefined precision/recall/F1 cases explicit as zero. Review ATTACK precision and recall together with the false-positive and false-negative counts: a lower false-positive rate can reduce benign alerts, while false negatives represent attack flows the model did not flag. The selected model follows the stated metric-based rule only; this benchmark result does not establish performance on modern live networks or production suitability.

### 2. Visualize held-out model comparisons and confusion matrices

Confusion matrices expose the false-positive and false-negative counts for both fitted models. The comparison chart displays all required scores, with false-positive rate included as an error-oriented metric. Random Forest feature importances are also plotted for up to the 20 most important transformed features; they describe the model's reliance on features, not causal effects or guaranteed attack indicators.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "macro_f1_score",
    "false_positive_rate",
]
metric_labels = ["Accuracy", "Attack precision", "Attack recall", "Attack F1", "Macro F1", "False-positive rate"]
model_colors = {"Logistic Regression": "#4C78A8", "Random Forest": "#F58518"}

for model_name, predictions in model_predictions.items():
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        labels=[0, 1],
        display_labels=["BENIGN", "ATTACK"],
        cmap="Blues",
        values_format="d",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{model_name}: held-out test confusion matrix")
    fig.tight_layout()
    confusion_path = FIGURE_DIR / f"confusion_matrix_{'logistic_regression' if model_name == 'Logistic Regression' else 'random_forest'}.png"
    fig.savefig(confusion_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {model_name} confusion matrix: {confusion_path.resolve()}")

comparison = results_table.loc[:, metric_columns]
ax = comparison.rename(columns=dict(zip(metric_columns, metric_labels))).T.plot(
    kind="bar",
    figsize=(11, 5.5),
    color=[model_colors[name] for name in comparison.index],
    width=0.78,
)
ax.set_title("Logistic Regression vs Random Forest — held-out test metrics")
ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelrotation=25)
ax.legend(title="Model")
fig = ax.get_figure()
fig.tight_layout()
comparison_path = FIGURE_DIR / "model_comparison.png"
fig.savefig(comparison_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved model-comparison chart: {comparison_path.resolve()}")

random_forest_pipeline = fitted_pipelines["Random Forest"]
random_forest_classifier = random_forest_pipeline.named_steps["classifier"]
transformed_feature_names = random_forest_pipeline.named_steps["preprocessing"].get_feature_names_out()
feature_importances = pd.Series(
    random_forest_classifier.feature_importances_,
    index=transformed_feature_names,
    name="importance",
).sort_values(ascending=False).head(20).sort_values()
fig, ax = plt.subplots(figsize=(10, max(5, 0.32 * len(feature_importances))))
feature_importances.plot(kind="barh", color="#54A24B", ax=ax)
ax.set_title("Random Forest — top 20 transformed-feature importances")
ax.set_xlabel("Mean decrease in impurity (model feature importance)")
ax.set_ylabel("Transformed feature")
fig.tight_layout()
importance_path = FIGURE_DIR / "random_forest_feature_importance.png"
fig.savefig(importance_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved Random Forest feature-importance chart: {importance_path.resolve()}")

The confusion-matrix plots and comparison chart are generated from the same held-out predictions used in the results table, and saved in `outputs/figures/`. In each matrix, the off-diagonal cells are false positives (BENIGN predicted ATTACK) and false negatives (ATTACK predicted BENIGN). The Random Forest importance plot covers the top 20 encoded features when fitted; importance is model-specific and should not be interpreted as causal evidence. Inspect actual plot outputs before drawing any dataset-specific conclusions.

### 3. Persist the selected pipeline and actual evaluation results

Only after both models complete the held-out evaluation, save the selected preprocessing-plus-model pipeline to `artifacts/` and write computed metrics, confusion counts, full classification reports, model parameters, selected dataset subset/sample details, and the split strategy to `outputs/reports/`. The JSON and CSV contain actual notebook-run values; no expected or illustrative scores are written.

In [ ]:
from datetime import datetime, timezone
import json
import joblib
import sklearn

ARTIFACT_DIR = Path("artifacts")
REPORT_DIR = Path("outputs/reports")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

selected_artifact_path = ARTIFACT_DIR / "netguard_selected_pipeline.joblib"
metrics_csv_path = REPORT_DIR / "model_metrics.csv"
classification_reports_path = REPORT_DIR / "classification_reports.txt"
evaluation_json_path = REPORT_DIR / "evaluation_results.json"

joblib.dump(selected_pipeline, selected_artifact_path)
results_table.to_csv(metrics_csv_path, index_label="model")
with classification_reports_path.open("w", encoding="utf-8") as report_file:
    for model_name, report in classification_reports.items():
        report_file.write(f"Classification report — {model_name}\n")
        report_file.write(report)
        report_file.write("\n")

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "binary network-flow classification",
    "target_mapping": {"BENIGN": 0, "all observed non-BENIGN non-empty labels": 1},
    "source_dataset": {
        "file_name": DATA_PATH.name,
        "configured_path": str(DATA_PATH.resolve()),
        "file_size_bytes": int(DATA_PATH.stat().st_size),
        "source_rows_scanned": int(total_source_rows),
        "eligible_rows_before_sampling": int(total_eligible_rows),
        "sampling_strategy": "seeded random-priority per-class reservoir; proportional to eligible class counts",
        "sample_cap": int(MAX_SAMPLE_ROWS),
        "sample_rows": int(len(sampled)),
        "sample_class_counts": {str(int(key)): int(value) for key, value in sampled["target"].value_counts().items()},
        "chunk_size": int(CHUNK_SIZE),
        "sample_random_state": int(RANDOM_STATE),
    },
    "split": {
        "strategy": "stratified random row-level train_test_split",
        "test_fraction": float(TEST_SIZE),
        "random_state": int(RANDOM_STATE),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "train_class_counts": {str(int(key)): int(value) for key, value in y_train.value_counts().items()},
        "test_class_counts": {str(int(key)): int(value) for key, value in y_test.value_counts().items()},
        "limitation": "related flows may occur in both partitions, potentially overestimating generalization",
    },
    "features": {
        "input_columns": list(feature_columns),
        "transformed_columns": list(transformed_feature_names),
        "excluded_columns_and_reasons": exclusion_reasons,
    },
    "imbalance_strategy": "estimator class weights; no resampling",
    "models": {
        model_name: {
            "parameters": fitted_pipelines[model_name].named_steps["classifier"].get_params(deep=False),
            "metrics_on_held_out_test": model_results[model_name],
            "classification_report": classification_reports[model_name],
        }
        for model_name in fitted_pipelines
    },
    "selection_criteria": "highest ATTACK F1, then highest ATTACK recall, then lowest false-positive rate; exact tie favors Logistic Regression",
    "selected_model": selected_model_name,
    "prediction_rule": "estimator default predict() decision rule",
    "versions": {"python": os.sys.version.split()[0], "pandas": pd.__version__, "scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
    "warnings": [
        "False positives are benign flows flagged as attacks and can create investigation workload.",
        "False negatives are attacks predicted benign and represent missed detections.",
        "CIC-IDS2017 results do not establish performance on modern live networks or production readiness.",
    ],
    "artifacts": {
        "selected_pipeline": str(selected_artifact_path),
        "metrics_csv": str(metrics_csv_path),
        "classification_reports": str(classification_reports_path),
    },
}
with evaluation_json_path.open("w", encoding="utf-8") as result_file:
    json.dump(metadata, result_file, indent=2, ensure_ascii=False)

print(f"Saved selected fitted pipeline: {selected_artifact_path.resolve()}")
print(f"Saved actual metric table: {metrics_csv_path.resolve()}")
print(f"Saved actual classification reports: {classification_reports_path.resolve()}")
print(f"Saved evaluation metadata and results: {evaluation_json_path.resolve()}")
print("Subset filename, source row count, bounded sampling strategy, class counts, split settings, and limitations are recorded in the JSON report.")

The saved joblib artifact includes both the fitted preprocessing transformer and selected classifier, preserving the feature transformations needed for later inference. The CSV, text report, and JSON are written only from the real values produced by this run. Check the recorded subset filename, scanned/eligible/sample counts, class balance, seed, and stratified row-level split before interpreting the scores. Related flows crossing the split remain a possible source of optimistic evaluation; test metrics are not a guarantee for modern live traffic.

## Phase 3 summary

Compare Logistic Regression and Random Forest using only predictions on the shared, stratified held-out test set. Report accuracy, ATTACK precision/recall/F1, macro F1, false-positive rate, and confusion counts with zero-division handling. Choose the selected pipeline by ATTACK F1, then recall, then false-positive rate; save the fitted preprocessing-plus-classifier artifact and the actual evaluation results for reproducibility. Interpret false positives as benign flows flagged and false negatives as missed attacks. CIC-IDS2017 is a dated benchmark, and its results do not establish performance on modern live networks or production readiness. Populate any summary conclusions only from the actual run outputs.